# Loss Functions Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: MSE and Its Gradient

In [ ]:
```python

def mse(predictions, targets):

    n = len(predictions)

    total = 0.0

    for p, t in zip(predictions, targets):

        total += (p - t) ** 2

    return total / n

def mse_gradient(predictions, targets):

    n = len(predictions)

    grads = []

    for p, t in zip(predictions, targets):

        grads.append(2.0 * (p - t) / n)

    return grads

In [ ]:
```

### Step 2: Binary Cross-Entropy

The log(0) problem is real. If the model predicts exactly 0 for a positive example, log(0) = negative infinity. Clipping prevents this.

In [ ]:
```python

import math

def binary_cross_entropy(predictions, targets, eps=1e-15):

    n = len(predictions)

    total = 0.0

    for p, t in zip(predictions, targets):

        p_clipped = max(eps, min(1 - eps, p))

        total += -(t * math.log(p_clipped) + (1 - t) * math.log(1 - p_clipped))

    return total / n

def bce_gradient(predictions, targets, eps=1e-15):

    grads = []

    for p, t in zip(predictions, targets):

        p_clipped = max(eps, min(1 - eps, p))

        grads.append(-(t / p_clipped) + (1 - t) / (1 - p_clipped))

    return grads

In [ ]:
```

### Step 3: Categorical Cross-Entropy with Softmax

Softmax converts raw logits to probabilities. Then we compute the cross-entropy against one-hot targets.

In [ ]:
```python

def softmax(logits):

    max_val = max(logits)

    exps = [math.exp(x - max_val) for x in logits]

    total = sum(exps)

    return [e / total for e in exps]

def categorical_cross_entropy(logits, target_index, eps=1e-15):

    probs = softmax(logits)

    p = max(eps, probs[target_index])

    return -math.log(p)

def cce_gradient(logits, target_index):

    probs = softmax(logits)

    grads = list(probs)

    grads[target_index] -= 1.0

    return grads

In [ ]:
```

The gradient of softmax + cross-entropy simplifies beautifully: it's just (predicted probability - 1) for the true class, and (predicted probability) for all other classes. This elegant simplification is not a coincidence -- it's why softmax and cross-entropy are paired.

### Step 4: Label Smoothing

In [ ]:
```python

def label_smoothed_cce(logits, target_index, num_classes, alpha=0.1, eps=1e-15):

    probs = softmax(logits)

    loss = 0.0

    for i in range(num_classes):

        if i == target_index:

            smooth_target = 1.0 - alpha + alpha / num_classes

        else:

            smooth_target = alpha / num_classes

        p = max(eps, probs[i])

        loss += -smooth_target * math.log(p)

    return loss

In [ ]:
```

### Step 5: Contrastive Loss (Simplified InfoNCE)

In [ ]:
```python

def cosine_similarity(a, b):

    dot = sum(x * y for x, y in zip(a, b))

    norm_a = math.sqrt(sum(x * x for x in a))

    norm_b = math.sqrt(sum(x * x for x in b))

    if norm_a < 1e-10 or norm_b < 1e-10:

        return 0.0

    return dot / (norm_a * norm_b)

def contrastive_loss(anchor, positive, negatives, temperature=0.07):

    sim_pos = cosine_similarity(anchor, positive) / temperature

    sim_negs = [cosine_similarity(anchor, neg) / temperature for neg in negatives]

    max_sim = max(sim_pos, max(sim_negs)) if sim_negs else sim_pos

    exp_pos = math.exp(sim_pos - max_sim)

    exp_negs = [math.exp(s - max_sim) for s in sim_negs]

    total_exp = exp_pos + sum(exp_negs)

    return -math.log(max(1e-15, exp_pos / total_exp))

In [ ]:
```

### Step 6: MSE vs Cross-Entropy on Classification

Train the same network from lesson 04 (circle dataset) with both loss functions. Watch cross-entropy converge faster.

In [ ]:
```python

import random

def sigmoid(x):

    x = max(-500, min(500, x))

    return 1.0 / (1.0 + math.exp(-x))

def make_circle_data(n=200, seed=42):

    random.seed(seed)

    data = []

    for _ in range(n):

        x = random.uniform(-2, 2)

        y = random.uniform(-2, 2)

        label = 1.0 if x * x + y * y < 1.5 else 0.0

        data.append(([x, y], label))

    return data

class LossComparisonNetwork:

    def __init__(self, loss_type="bce", hidden_size=8, lr=0.1):

        random.seed(0)

        self.loss_type = loss_type

        self.lr = lr

        self.hidden_size = hidden_size

        self.w1 = [[random.gauss(0, 0.5) for _ in range(2)] for _ in range(hidden_size)]

        self.b1 = [0.0] * hidden_size

        self.w2 = [random.gauss(0, 0.5) for _ in range(hidden_size)]

        self.b2 = 0.0

    def forward(self, x):

        self.x = x

        self.z1 = []

        self.h = []

        for i in range(self.hidden_size):

            z = self.w1[i][0] * x[0] + self.w1[i][1] * x[1] + self.b1[i]

            self.z1.append(z)

            self.h.append(max(0.0, z))

        self.z2 = sum(self.w2[i] * self.h[i] for i in range(self.hidden_size)) + self.b2

        self.out = sigmoid(self.z2)

        return self.out

    def backward(self, target):

        if self.loss_type == "mse":

            d_loss = 2.0 * (self.out - target)

        else:

            eps = 1e-15

            p = max(eps, min(1 - eps, self.out))

            d_loss = -(target / p) + (1 - target) / (1 - p)

        d_sigmoid = self.out * (1 - self.out)

        d_out = d_loss * d_sigmoid

        for i in range(self.hidden_size):

            d_relu = 1.0 if self.z1[i] > 0 else 0.0

            d_h = d_out * self.w2[i] * d_relu

            self.w2[i] -= self.lr * d_out * self.h[i]

            for j in range(2):

                self.w1[i][j] -= self.lr * d_h * self.x[j]

            self.b1[i] -= self.lr * d_h

        self.b2 -= self.lr * d_out

    def compute_loss(self, pred, target):

        if self.loss_type == "mse":

            return (pred - target) ** 2

        else:

            eps = 1e-15

            p = max(eps, min(1 - eps, pred))

            return -(target * math.log(p) + (1 - target) * math.log(1 - p))

    def train(self, data, epochs=200):

        losses = []

        for epoch in range(epochs):

            total_loss = 0.0

            correct = 0

            for x, y in data:

                pred = self.forward(x)

                self.backward(y)

                total_loss += self.compute_loss(pred, y)

                if (pred >= 0.5) == (y >= 0.5):

                    correct += 1

            avg_loss = total_loss / len(data)

            accuracy = correct / len(data) * 100

            losses.append((avg_loss, accuracy))

            if epoch % 50 == 0 or epoch == epochs - 1:

                print(f"    Epoch {epoch:3d}: loss={avg_loss:.4f}, accuracy={accuracy:.1f}%")

        return losses

In [ ]:
```

## Exercises

In [ ]:
1. Implement Huber loss (smooth L1 loss), which is MSE for small errors and MAE for large errors. Train a regression network predicting y = sin(x) with MSE vs Huber when 5% of training targets have random noise added (outliers). Compare final test error.

2. Add focal loss to the binary classification training loop. Create an imbalanced dataset (90% class 0, 10% class 1). Compare standard BCE vs focal loss (gamma=2) on the minority class recall after 200 epochs.

3. Implement triplet loss with semi-hard negative mining. Generate 2D embedding data for 5 classes. For each anchor, find the hardest negative that is still farther than the positive (semi-hard). Compare convergence to random triplet selection.

4. Run the MSE vs cross-entropy comparison but track gradient magnitudes at each layer during training. Plot the average gradient norm per epoch. Verify that cross-entropy produces larger gradients in early epochs when the model is most uncertain.

5. Implement KL divergence loss and verify that minimizing KL(true || predicted) gives the same gradients as cross-entropy when the true distribution is one-hot. Then try soft targets (like knowledge distillation) where the "true" distribution comes from a teacher model's softmax output.